In [1]:
from models_v3 import *
from torch.utils.data import DataLoader
from torch import nn
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
from torchvision.transforms import v2
from medmnist import PathMNIST

tf = v2.Compose([
    v2.ToTensor()
])

test_dataset = PathMNIST(root="./data/",split="val",transform=tf,download=True,size=64)

n_labels = len(test_dataset.info["label"].items())

/mnt/sdc/Hasan/Documents/My Documents/Code/MAE-Model-MedMNIST-Predictor/.venv/lib64/python3.14/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [3]:

num_workers = 4

test_batch_size = 128

test_dl = DataLoader(
    test_dataset,
    batch_size= test_batch_size,
    num_workers=num_workers,
    pin_memory=True,
)

In [ ]:
#ae = AutoEncoder(
#    patcher=CNN(torch.load("model_weights/pretraining/patcher.pt")),
#    encoder=Encoder(torch.load("model_weights/pretraining/encoder.pt")))
#ae.pat_to_enc.load_state_dict(torch.load("model_weights/pretraining/linear_pte.pt"))
#model = Predictor(autoencoder=ae,n_labels=n_labels)

model = Predictor(n_labels=n_labels)
model.load_state_dict(torch.load("model_weights/checkpoints/model_2_epoch_20.pt"))

<All keys matched successfully>

In [5]:
loss = nn.CrossEntropyLoss(label_smoothing=0.1)

In [6]:
from training_functions import test
import csv
import time

log_file_path = f"logs/v3_testing_log_{time.time()}.csv"

with open(log_file_path, mode="w", newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["epoch", "test_loss", "test_acc","test_auc"])
    model = model.to(device)
    torch.set_float32_matmul_precision('high')
    model = torch.compile(model)
    test_loss,test_acc,test_auc = test(model, device, test_dl, loss, 0,"Testing")
    print(
        f"_______________________________________________\n"
        f"Results:\n"
        f"    Loss: {test_loss:4f}\n"
        f"    Accuracy: {test_acc:4f}\n"
        f"    AUC: {test_auc:4f}\n"
        "_______________________________________________\n")
    writer.writerow([1, test_loss, test_acc,test_auc])
    f.flush()

Epoch 1: Testing:   0%|          | 0/79 [00:00<?, ?it/s]

_______________________________________________
Results:
    Loss: 0.508316
    Accuracy: 0.990804
    AUC: 0.999829
_______________________________________________

